# Dados agrícolas do IBGE para o estudo de crédito rural

Este notebook prepara dados anuais da Produção Agrícola Municipal (PAM), tabela SIDRA 1612, para soja em grão, milho em grão e cana-de-açúcar. O código atual prevê o período de 2016 a 2025. Esta documentação descreve cada etapa e registra pendências observadas na revisão de 20/10/2026; não apresenta resultados analíticos validados.

As perguntas de referência são:

| Referência | Pergunta de domínio |
|---|---|
| P1 | Como evoluíram o valor contratado e o número de contratos de custeio por cultura na década? |
| P2 | Como o crédito e a área plantada se distribuem entre estados e regiões? |
| P3 | O crédito está mais concentrado em poucos municípios do que a produção? |
| P4 | Como variam o crédito e os contratos entre municípios com áreas plantadas semelhantes? |
| P5 | Como variam o crédito por tonelada e por hectare entre municípios e estados? |
| P6 | Municípios com maior crédito por hectare apresentam distribuições diferentes de produtividade, na mesma cultura e ano? |
| P7 | Quais municípios apresentam valores atípicos de crédito relativo ou produtividade no padrão dos estados selecionados? |

**Escopo:** a tabela 1612 fornece medidas agrícolas. Não contém o valor contratado nem a quantidade de contratos de crédito, portanto não responde diretamente a P1 nem à parte financeira das demais perguntas. A indicação de uma pergunta em cada célula identifica apenas sua contribuição agrícola.

**Leitura das saídas:** há resultados salvos incompatíveis com o código atual. Eles foram preservados nesta revisão documental e não devem ser tratados como comprovação das consultas modificadas.

### Tratamento dos valores e cobertura

O código converte `-` em zero, aceita textos compostos por dígitos e os transforma em inteiros; os demais registros são descartados. Na convenção do IBGE, `..` indica não aplicável; `...`, indisponível; e `X`, informação omitida.

**Por que esta regra precisa ser documentada:** zero e ausência têm significados diferentes. O descarte pode alterar o conjunto de municípios usado nas perguntas P2 a P7. É necessário preservar o símbolo original ou registrar o motivo e a quantidade de exclusões por variável, cultura e ano. `isdecimal()` não é um teste numérico geral: valores com parte decimal também seriam descartados.

### 1. Bibliotecas de trabalho

Importa as bibliotecas usadas pelo notebook. `pandas` organiza as respostas em tabelas e `requests` realiza as consultas HTTP; `numpy` e `altair` estão importadas, mas não são utilizadas nas extrações atuais.

**Por que é necessária:** prepara as ferramentas comuns às consultas. **Perguntas apoiadas:** P2 a P7, como infraestrutura; esta célula não produz uma resposta de domínio.

In [38]:
import pandas as pd
import numpy as np
import altair as alt
import requests as rqs

### 2. Validação do certificado HTTPS

Define o adaptador `IBGETLSAdapter`, que complementa a cadeia de certificados do IBGE com o intermediário local, mantendo a verificação do domínio, da validade e da cadeia até uma raiz confiável.

**Por que é necessária:** permite consultar o serviço com validação TLS ativa diante da cadeia incompleta identificada no servidor. **Perguntas apoiadas:** P2 a P7, indiretamente, viabilizando a coleta. Requer o arquivo de certificado indicado na configuração.

In [39]:
import ssl
from pathlib import Path
from requests.adapters import HTTPAdapter


class IBGETLSAdapter(HTTPAdapter):
    """Completa a cadeia do IBGE sem desativar a valida??o TLS."""

    def __init__(self, certificado_intermediario, **kwargs):
        self.ssl_context = ssl.create_default_context(cafile=rqs.certs.where())
        # O intermedi?rio deve encadear at? uma raiz confi?vel, sem cadeia parcial.
        self.ssl_context.verify_flags &= ~ssl.VERIFY_X509_PARTIAL_CHAIN
        self.ssl_context.load_verify_locations(cafile=str(certificado_intermediario))
        super().__init__(**kwargs)

    def init_poolmanager(self, connections, maxsize, block=False, **pool_kwargs):
        pool_kwargs['ssl_context'] = self.ssl_context
        return super().init_poolmanager(connections, maxsize, block=block, **pool_kwargs)

    def proxy_manager_for(self, proxy, **proxy_kwargs):
        proxy_kwargs['ssl_context'] = self.ssl_context
        return super().proxy_manager_for(proxy, **proxy_kwargs)

### 3. Identificadores das culturas e variáveis

Centraliza os códigos de soja, milho e cana-de-açúcar, da tabela 1612, das variáveis agrícolas e do certificado. Área plantada (`109`) e colhida (`216`) usam hectares; produção (`214`), toneladas; rendimento (`112`), kg/ha. Os códigos percentuais não representam hectares.

**Por que é necessária:** torna as escolhas da coleta explícitas e evita repetir números sem identificação. **Perguntas apoiadas:** P2 a P7. Declarar uma variável não significa que seus dados já foram extraídos; a área colhida ainda não tem consulta.

In [49]:
id_milho = 2711
id_soja = 2713
id_cana = 2696
#Área plantada Hectares
area_plant_H = 109
#Área plantada - percentual do total geral %
area_plant_100 = 1000109
#Área colhida Hectares
area_colhida_H = 216
#Área colhida - percentual do total geral %
area_colhida_100 = 1000216
#Quantidade produzida Toneladas
quant_prod_T = 214
#Rendimento médio da produção Quilogramas por Hectare
rend_medio_Kg_H = 112
#Valor da produção - percentual do total geral %
tabela_lavouras_temporárias = 1612

parametros = {
    "localidades": "N2[all]",
    "classificacao": f"81[{id_soja},{id_cana},{id_milho}]"
}

certificado_intermediario = Path("certs/SectigoPublicServerAuthenticationCADVR36.pem")

### 4. Função de requisição certificada

Encapsula a abertura da sessão HTTPS, o envio dos parâmetros, o limite de espera, a verificação do status HTTP e a leitura do JSON.

**Por que é necessária:** aplica o mesmo procedimento de acesso às diferentes consultas, reduzindo repetição. **Perguntas apoiadas:** P2 a P7, como infraestrutura.

**Pontos de revisão:** o retorno pode ser um objeto ou uma lista, embora a anotação indique apenas `dict`; o argumento padrão `{}` é mutável. A função também depende do certificado definido anteriormente.

In [41]:
def requisicao_certificada(url: str, parametros: dict = {}) -> dict:
    with rqs.Session() as sessao:
        sessao.mount("https://servicodados.ibge.gov.br/", IBGETLSAdapter(certificado_intermediario))
        response = sessao.get(url, params=parametros, timeout=30)
        response.raise_for_status()
        return response.json()

### 5. Conferência dos metadados da tabela

Consulta os nomes, códigos e unidades das variáveis e as categorias da classificação de produtos das lavouras temporárias.

**Por que é necessária:** permite conferir se os identificadores usados correspondem às culturas e medidas pretendidas. **Perguntas apoiadas:** P2 a P7, validando a definição dos dados. A exibição dos metadados não comprova a cobertura dos anos ou municípios das consultas seguintes.

In [42]:

url_metadados = f"https://servicodados.ibge.gov.br/api/v3/agregados/{tabela_lavouras_temporárias}/metadados"

dados = requisicao_certificada(url_metadados)

print("Tabela:", dados["nome"])

for variavel in dados["variaveis"]:
    print(
        variavel["id"],
        variavel["nome"],
        variavel["unidade"]
    )

for classificacao in dados["classificacoes"]:
    print("Classificação:", classificacao["id"], classificacao["nome"])

    for categoria in classificacao["categorias"]:
        print(categoria["id"], categoria["nome"])

for classificacao in dados["classificacoes"]:
    print(classificacao["id"], classificacao["nome"])

Tabela: Área plantada, área colhida, quantidade produzida, rendimento médio e valor da produção das lavouras temporárias
109 Área plantada Hectares
1000109 Área plantada - percentual do total geral %
216 Área colhida Hectares
1000216 Área colhida - percentual do total geral %
214 Quantidade produzida Toneladas
112 Rendimento médio da produção Quilogramas por Hectare
215 Valor da produção Mil Cruzeiros [1974 a 1985, 1990 a 1992], Mil Cruzados [1986 a 1988], Mil Cruzados Novos [1989], Mil Cruzeiros Reais [1993], Mil Reais [1994 a 2025]
1000215 Valor da produção - percentual do total geral %
Classificação: 81 Produto das lavouras temporárias
0 Total
2688 Abacaxi*
83374 Abóbora
40471 Alfafa fenada
83375 Alface
2689 Algodão herbáceo (em caroço)
2690 Alho
2691 Amendoim (em casca)
2692 Arroz (em casca)
2693 Aveia (em grão)
2694 Batata-doce
2695 Batata-inglesa
2696 Cana-de-açúcar
40470 Cana para forragem
83384 Canola
2697 Cebola
83381 Cenoura
2698 Centeio (em grão)
2699 Cevada (em grão)
83379 

### 6. Área plantada por região

Solicita área plantada em hectares (`109`), para as três culturas, nas Grandes Regiões (`N2`), entre 2016 e 2025, e pretende organizar o resultado em `df_area_regional`.

**Por que é necessária:** fornece a medida agrícola para descrever a distribuição regional de cada cultura. **Pergunta apoiada:** P2, na parte de área plantada.

**Erro a corrigir:** os registros criam `codigo_municipio` e `municipio`, mas a ordenação exige `regiao`. Os campos precisam identificar regiões; com dados retornados, a ordenação atual falha. A saída salva usa colunas diferentes das criadas pelo código atual.

In [ ]:
periodo = "2016-2025"


url = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/"
    f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{area_plant_H}"
)

dados = requisicao_certificada(url, parametros=parametros)

variavel = dados[0]
resultado = variavel["resultados"][0]

linhas_area_regional = []

for resultado in variavel["resultados"]:
    categorias = resultado["classificacoes"][0]["categoria"]

    for codigo_cultura, nome_cultura in categorias.items():
        for serie in resultado["series"]:
            for ano, valor in serie["serie"].items():
                if valor == "-":
                    valor = 0
                elif not valor.isdecimal():
                    continue
                linhas_area_regional.append({
                    "codigo_cultura": codigo_cultura,
                    "cultura": nome_cultura,
                    "codigo_regiao": serie["localidade"]["id"],
                    "regiao": serie["localidade"]["nome"],
                    "ano": int(ano),
                    "variavel": variavel["variavel"],
                    "unidade": variavel["unidade"],
                    "valor": int(valor)
                })

df_area_regional = pd.DataFrame(linhas_area_regional)
df_area_regional = df_area_regional.sort_values(["regiao","cultura","ano","valor"])

df_area_regional.head()

,codigo_cultura,cultura,codigo_regiao,regiao,ano,variavel,unidade,valor
90,2696,Cana-de-açúcar,5,Centro-Oeste,2016,Área plantada,Hectares,1870011
91,2696,Cana-de-açúcar,5,Centro-Oeste,2017,Área plantada,Hectares,1864280
92,2696,Cana-de-açúcar,5,Centro-Oeste,2018,Área plantada,Hectares,1914387
93,2696,Cana-de-açúcar,5,Centro-Oeste,2019,Área plantada,Hectares,1972043
94,2696,Cana-de-açúcar,5,Centro-Oeste,2020,Área plantada,Hectares,1859847


### 7. Área plantada por estado

Altera o nível territorial para Unidades da Federação (`N3`) e pretende obter a área plantada das três culturas no mesmo intervalo regional, formando `df_area_estadual`.

**Por que é necessária:** fornece a distribuição estadual da área e a medida de hectares usada nas comparações estaduais. **Perguntas apoiadas:** P2 e P5, na parte agrícola.

**Erros a corrigir:** os registros usam nomes de campos municipais, mas a ordenação exige `estado`. Além disso, a célula herda `url` de outra consulta; ao reexecutá-la fora de ordem, pode solicitar outra variável ou período.

In [44]:
#foi verificado que se o campo valor fosse nulo ele poderia ser "..." ou "-" logo o if valor.isdecimal é valido

parametros["localidades"] = "N3[all]"

dados = requisicao_certificada(url, parametros=parametros)

variavel = dados[0]
resultado = variavel["resultados"][0]

linhas_area_estadual = []

for resultado in variavel["resultados"]:
    categorias = resultado["classificacoes"][0]["categoria"]

    for codigo_cultura, nome_cultura in categorias.items():
        for serie in resultado["series"]:
            for ano, valor in serie["serie"].items():
                if valor == "-":
                    valor = 0
                elif not valor.isdecimal():
                    continue
                linhas_area_estadual.append({
                    "codigo_cultura": codigo_cultura,
                    "cultura": nome_cultura,
                    "codigo_estado": serie["localidade"]["id"],
                    "estado": serie["localidade"]["nome"],
                    "ano": int(ano),
                    "variavel": variavel["variavel"],
                    "unidade": variavel["unidade"],
                    "valor": int(valor)
                })
            

df_area_estadual = pd.DataFrame(linhas_area_estadual)
df_area_estadual = df_area_estadual.sort_values(["estado","cultura","ano","valor"])

df_area_estadual.head()

,codigo_cultura,cultura,codigo_estado,estado,ano,variavel,unidade,valor
280,2696,Cana-de-açúcar,12,Acre,2016,Área plantada,Hectares,3079
281,2696,Cana-de-açúcar,12,Acre,2017,Área plantada,Hectares,2437
282,2696,Cana-de-açúcar,12,Acre,2018,Área plantada,Hectares,517
283,2696,Cana-de-açúcar,12,Acre,2019,Área plantada,Hectares,513
284,2696,Cana-de-açúcar,12,Acre,2020,Área plantada,Hectares,490


### 8. Quantidade produzida por município

Solicita produção em toneladas (`214`) nos municípios (`N6`), ano a ano de 2016 a 2025, para as três culturas, e reúne os registros em `df_prod_municipal`.

**Por que é necessária:** fornece o volume produzido em cada município para estudar sua concentração e os indicadores relativos à produção. **Perguntas apoiadas:** P3 e P5; também auxilia a conferência agrícola de P7.

**Pontos de revisão:** a saída salva ainda mostra 2015 e não corresponde ao intervalo atual. O bloco seguinte sobrescreve este DataFrame. Os parâmetros de cultura continuam sendo herdados de uma célula anterior.

In [45]:
parametros["localidades"] = "N6[all]"
linhas_quant_municipal = []
for periodo in range(2016,2026):
    url = (
        "https://servicodados.ibge.gov.br/api/v3/agregados/"
        f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{quant_prod_T}"
    )

    dados = requisicao_certificada(url, parametros=parametros)

    variavel = dados[0]
    resultado = variavel["resultados"][0]

    for resultado in variavel["resultados"]:
        categorias = resultado["classificacoes"][0]["categoria"]

        for codigo_cultura, nome_cultura in categorias.items():
            for serie in resultado["series"]:
                for ano, valor in serie["serie"].items():
                    if valor == "-":
                        valor = 0
                    elif not valor.isdecimal():
                        continue
                    linhas_quant_municipal.append({
                        "codigo_cultura": codigo_cultura,
                        "cultura": nome_cultura,
                        "codigo_municipio": serie["localidade"]["id"],
                        "municipio": serie["localidade"]["nome"],
                        "ano": int(ano),
                        "variavel": variavel["variavel"],
                        "unidade": variavel["unidade"],
                        "valor": int(valor)
                    })
                

df_prod_municipal = pd.DataFrame(linhas_quant_municipal)
df_prod_municipal = df_prod_municipal.sort_values(["municipio","cultura","ano","valor"])

df_prod_municipal.head()

,codigo_cultura,cultura,codigo_municipio,municipio,ano,variavel,unidade,valor
10717,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2016,Quantidade produzida,Toneladas,0
27179,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2017,Quantidade produzida,Toneladas,300
43658,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2018,Quantidade produzida,Toneladas,238
60144,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2019,Quantidade produzida,Toneladas,1208
76618,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2020,Quantidade produzida,Toneladas,0


### 9. Quantidade produzida por estado

Solicita produção em toneladas (`214`) por Unidade da Federação (`N3`), de 2016 a 2025, para as culturas selecionadas.

**Por que é necessária:** fornece o volume estadual usado como denominador dos indicadores por tonelada. **Pergunta apoiada:** P5, na parte agrícola estadual.

**Erros a corrigir:** os campos territoriais estão nomeados como municipais e o resultado é atribuído a `df_prod_municipal`, substituindo os dados municipais da célula anterior. A extração precisa identificar estados e manter seu próprio DataFrame.

In [46]:
parametros["localidades"] = "N3[all]"
linhas_quant_estadual = []
periodo = "2016-2025"
url = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/"
    f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{quant_prod_T}"
)

dados = requisicao_certificada(url, parametros=parametros)

variavel = dados[0]
resultado = variavel["resultados"][0]

for resultado in variavel["resultados"]:
    categorias = resultado["classificacoes"][0]["categoria"]

    for codigo_cultura, nome_cultura in categorias.items():
        for serie in resultado["series"]:
            for ano, valor in serie["serie"].items():
                if valor == "-":
                    valor = 0
                elif not valor.isdecimal():
                    continue
                linhas_quant_estadual.append({
                    "codigo_cultura": codigo_cultura,
                    "cultura": nome_cultura,
                    "codigo_estado": serie["localidade"]["id"],
                    "estado": serie["localidade"]["nome"],
                    "ano": int(ano),
                    "variavel": variavel["variavel"],
                    "unidade": variavel["unidade"],
                    "valor": int(valor)
                })
                

df_prod_estadual = pd.DataFrame(linhas_quant_estadual)
df_prod_estadual = df_prod_estadual.sort_values(["estado","cultura","ano","valor"])

df_prod_estadual.head()

,codigo_cultura,cultura,codigo_estado,estado,ano,variavel,unidade,valor
280,2696,Cana-de-açúcar,12,Acre,2016,Quantidade produzida,Toneladas,117244
281,2696,Cana-de-açúcar,12,Acre,2017,Quantidade produzida,Toneladas,20229
282,2696,Cana-de-açúcar,12,Acre,2018,Quantidade produzida,Toneladas,11989
283,2696,Cana-de-açúcar,12,Acre,2019,Quantidade produzida,Toneladas,12119
284,2696,Cana-de-açúcar,12,Acre,2020,Quantidade produzida,Toneladas,11822


### 10. Área plantada por município

Solicita área plantada em hectares (`109`) nos municípios (`N6`), para as três culturas e cada ano de 2016 a 2025, formando `df_area_municipal`.

**Por que é necessária:** caracteriza a escala de cultivo de cada município e fornece a área para indicadores por hectare. **Perguntas apoiadas:** P4 e P5; auxilia a identificação de denominadores pequenos em P7.

**Pontos de revisão:** conferir a cobertura anual, os registros descartados e a existência de duplicatas. A ordem da saída salva não corresponde à ordenação atual por município, cultura e ano.

In [47]:
parametros["localidades"] = "N6[all]"
linhas_area_municipal = []
for periodo in range(2016,2026):
    url = (
        "https://servicodados.ibge.gov.br/api/v3/agregados/"
        f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{area_plant_H}"
    )

    dados = requisicao_certificada(url, parametros=parametros)

    variavel = dados[0]
    resultado = variavel["resultados"][0]

    for resultado in variavel["resultados"]:
        categorias = resultado["classificacoes"][0]["categoria"]

        for codigo_cultura, nome_cultura in categorias.items():
            for serie in resultado["series"]:
                for ano, valor in serie["serie"].items():
                    if valor == "-":
                        valor = 0
                    elif not valor.isdecimal():
                        continue

                    linhas_area_municipal.append({
                        "codigo_cultura": codigo_cultura,
                        "cultura": nome_cultura,
                        "codigo_municipio": serie["localidade"]["id"],
                        "municipio": serie["localidade"]["nome"],
                        "ano": int(ano),
                        "variavel": variavel["variavel"],
                        "unidade": variavel["unidade"],
                        "valor": int(valor)
                    })
                    

df_area_municipal = pd.DataFrame(linhas_area_municipal)

df_area_municipal = df_area_municipal.sort_values(["municipio","cultura","ano","valor"])
df_area_municipal.head()

,codigo_cultura,cultura,codigo_municipio,municipio,ano,variavel,unidade,valor
10717,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2016,Área plantada,Hectares,0
27179,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2017,Área plantada,Hectares,5
43658,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2018,Área plantada,Hectares,4
60144,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2019,Área plantada,Hectares,15
76618,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2020,Área plantada,Hectares,0


### 11. Rendimento médio por município

Solicita o rendimento médio oficial (`112`), em kg/ha, para as três culturas, nos municípios (`N6`), de 2016 a 2025, formando `df_rendimento_municipal`.

**Por que é necessária:** fornece a produtividade municipal para examinar sua distribuição e valores atípicos. **Perguntas apoiadas:** P6 e P7, na parte agrícola.

**Pontos de revisão:** a saída salva mostra área plantada em hectares, e não rendimento, portanto precisa ser atualizada após a execução correta. A produtividade oficial se refere à área colhida; consultar `216` é recomendado para conferir sua coerência com a produção.

In [48]:
parametros["localidades"] = "N6[all]"
linhas_rendimento = []
for periodo in range(2016,2026):
    url = (
        "https://servicodados.ibge.gov.br/api/v3/agregados/"
        f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{rend_medio_Kg_H}"
    )

    dados = requisicao_certificada(url, parametros=parametros)

    variavel = dados[0]
    resultado = variavel["resultados"][0]

    for resultado in variavel["resultados"]:
        categorias = resultado["classificacoes"][0]["categoria"]

        for codigo_cultura, nome_cultura in categorias.items():
            for serie in resultado["series"]:
                for ano, valor in serie["serie"].items():
                    if valor == "-":
                        valor = 0
                    elif not valor.isdecimal():
                        continue
                    linhas_rendimento.append({
                        "codigo_cultura": codigo_cultura,
                        "cultura": nome_cultura,
                        "codigo_municipio": serie["localidade"]["id"],
                        "municipio": serie["localidade"]["nome"],
                        "ano": int(ano),
                        "variavel": variavel["variavel"],
                        "unidade": variavel["unidade"],
                        "valor": int(valor)
                    })

df_rendimento_municipal = pd.DataFrame(linhas_rendimento)

df_rendimento_municipal = df_rendimento_municipal.sort_values(["municipio","cultura","ano","valor"])
df_rendimento_municipal.head()

,codigo_cultura,cultura,codigo_municipio,municipio,ano,variavel,unidade,valor
10717,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2016,Rendimento médio da produção,Quilogramas por Hectare,0
27179,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2017,Rendimento médio da produção,Quilogramas por Hectare,60000
43658,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2018,Rendimento médio da produção,Quilogramas por Hectare,59500
60144,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2019,Rendimento médio da produção,Quilogramas por Hectare,80533
76618,2696,Cana-de-açúcar,5200050,Abadia de Goiás - GO,2020,Rendimento médio da produção,Quilogramas por Hectare,0


## Pendências da coleta e sequência de verificação

A revisão examinou o código e as saídas salvas; não executou as consultas nem alterou células de código.

| Prioridade | Pendência | Por que é necessária |
|---|---|---|
| 1 | Corrigir os campos territoriais das áreas regional e estadual. | As ordenações usam colunas que o código não cria e impedem uma execução completa. |
| 1 | Identificar corretamente a produção estadual e separá-la da municipal. | O bloco estadual substitui `df_prod_municipal` e perde a distinção territorial no nome dos campos. |
| 2 | Explicitar variável, período, culturas e território em cada chamada. | A reutilização de `url` e a alteração do mesmo `parametros` tornam o resultado dependente da ordem de execução. |
| 2 | Registrar os valores especiais e os descartes. | A exclusão silenciosa impede avaliar a cobertura dos municípios e anos. |
| 2 | Atualizar as saídas após corrigir o código e executar desde um kernel reiniciado. | Existem saídas de 2015 em consultas de 2016–2025 e uma saída de área no bloco de rendimento. |
| 3 | Definir os estados do recorte e conferir a cobertura retornada. | As consultas usam `all`, mas P7 menciona estados selecionados. Consultar o país inteiro é válido se esse for o recorte declarado. |
| 3 | Considerar a coleta de área colhida (`216`). | Permite conferir a coerência do rendimento com a quantidade produzida; não é obrigatória para usar o rendimento oficial `112`. |
| 3 | Registrar períodos disponíveis, URLs, parâmetros, unidades e data de coleta. | Permite rastrear e reproduzir as extrações. |

Após essas correções, conferir se as três culturas, os dez anos e as localidades esperadas estão presentes, se há duplicatas por localidade–cultura–ano–variável e se as unidades correspondem à medida solicitada. Uma prévia com `head()` não valida a cobertura completa.

## Funções e independência das células

**Sim, funções são adequadas para este notebook.** Elas podem concentrar a requisição certificada, a montagem da consulta, a leitura da resposta e o tratamento documentado dos valores, reduzindo as cópias dos mesmos laços.

Aqui, independência significa que cada célula de extração declara os parâmetros da sua consulta e recebe seu próprio resultado, sem depender do último valor de `url`, `periodo` ou `parametros` deixado por outra extração. É normal que todas dependam das células iniciais de importação, configuração e definição de funções.

Para este projeto, a melhor organização é manter funções pequenas com responsabilidades claras e deixar cada consulta visível no corpo do notebook. Uma função de extração deve receber explicitamente tabela, variável, período, território e culturas, criar seus acumuladores internamente e devolver um novo DataFrame. A política de valores ausentes deve ser explícita e rastreável. Assim, cada etapa do artigo continua legível e a lógica repetida fica centralizada.

Nesta alteração, essa organização foi apenas descrita: nenhuma função foi criada ou modificada.

Fontes metodológicas: [API de Agregados do IBGE](https://servicodados.ibge.gov.br/api/docs/agregados?versao=3) e [definições de rendimento médio do IBGE](https://anuario.ibge.gov.br/2024/glossario-2024.html).